# 🤖 MBTI Personality Detection — RAG + LLM Agent (LangChain)

---

**Architecture: Agent = LLM + Planning + Tool Use + Loop + Memory**

| Component | Implementation |
|-----------|---------------|
| **Framework** | **LangChain** — Agent, Tools, VectorStore, Embeddings, LLM integration |
| **LLM** | Qwen2.5-7B-Instruct (4-bit quantization) via `HuggingFacePipeline` |
| **Planning** | LangChain `create_react_agent` — ReAct prompt-based autonomous reasoning |
| **Tool Use** | LangChain `@tool` decorator — structured tool definitions with automatic schema |
| **Loop** | `AgentExecutor` with configurable `max_iterations` and `early_stopping_method` |
| **Memory** | Short-term: `AgentExecutor` conversation history \| Long-term: FAISS experience store |
| **RAG** | LangChain `FAISS` vectorstore + `HuggingFaceEmbeddings` |
| **Guardrail** | NLI Post-Verification — independent DeBERTa-v3-base verifies entailment per dimension |

**RAG (Retrieval-Augmented Generation):**
- **VectorStore 1 — Similar Posts**: Training posts → `FAISS` vectorstore → few-shot examples via `as_retriever()`
- **VectorStore 2 — MBTI Knowledge Base**: Dimension descriptions + 16-type profiles → `FAISS` vectorstore

**NLI Post-Verification Guardrail (Hậu kiểm):**
- After the agent predicts, an **independent NLI model** (DeBERTa-v3-base, ~184M params) verifies each dimension
- Constructs (premise=posts, hypothesis=predicted trait claim) and checks P(entailment)
- Rejects when P(entailment) < threshold — agent must revise
- After `MAX_GUARDRAIL_RETRIES` rejections, force-accept to prevent infinite loops

**LangChain Components Used:**
- `langchain_huggingface.HuggingFaceEmbeddings` — embedding model wrapper
- `langchain_huggingface.HuggingFacePipeline` — LLM wrapper
- `langchain_community.vectorstores.FAISS` — vector store + retriever
- `langchain.tools.tool` — tool decorator with auto JSON schema
- `langchain.agents.create_react_agent` — ReAct agent factory
- `langchain.agents.AgentExecutor` — agent execution loop with memory
- `sentence_transformers.CrossEncoder` — NLI cross-encoder for guardrail

**Datasets supported:**
- MBTI-500 (~106K rows): `/kaggle/input/.../MBTI 500.csv`
- Kaggle MBTI (~8600 rows): `/kaggle/input/.../mbti_1.csv`

## 📦 Cell 1 — Install Dependencies

In [ ]:
# ⚠️ Do NOT install/upgrade torch or transformers — Kaggle has CUDA 12-compatible versions pre-installed.
!pip install -q --no-deps bitsandbytes accelerate
!pip install -q faiss-cpu
!pip install -q "sentence-transformers>=3.0,<4.0"
!pip install -q "langchain>=0.3.0,<0.4.0" "langchain-community>=0.3.0,<0.4.0" "langchain-core>=0.3.0,<0.4.0" langchain-huggingface

# ── Verify torch was NOT upgraded to CUDA 13 ──
import subprocess, sys
_tv = subprocess.check_output([sys.executable, '-c',
    'import torch; print(torch.__version__)']).decode().strip()
print(f'✅ torch version: {_tv}')
if 'cu130' in _tv or 'cu131' in _tv:
    print('❌ ERROR: torch was upgraded to CUDA 13 but Kaggle only has CUDA 12!')
    print('   → Go to Runtime > Restart session, then re-run ALL cells.')
    raise RuntimeError(f'torch {_tv} requires CUDA 13. Restart kernel to restore Kaggle defaults.')

## 📥 Cell 2 — Imports

In [ ]:
import os, re, json, time, warnings, gc
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import faiss

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers import pipeline as hf_pipeline

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    classification_report
)
from sklearn.preprocessing import label_binarize

# ── LangChain imports ──
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS as LangChainFAISS
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain.agents import create_react_agent, AgentExecutor

import langchain
print(f'✅ Imports done (LangChain {langchain.__version__} + HuggingFace)')

warnings.filterwarnings('ignore')

# Suppress HuggingFace "using pipelines sequentially on GPU" warning
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
import logging
logging.getLogger('transformers.pipelines').setLevel(logging.ERROR)

## ⚙️ Cell 3 — Config

In [ ]:
# ───────────────────────────────────────────────────────────
# CONFIG — all paths and hyperparameters here
# ───────────────────────────────────────────────────────────
CONFIG = {
    # Paths
    'DATA_PATH_500':  '/kaggle/input/datasets/zeyadkhalid/mbti-personality-types-500-dataset/MBTI 500.csv',
    'DATA_PATH_8K':   '/kaggle/input/datasets/datasnaek/mbti-type/mbti_1.csv',
    'RESULT_DIR':     '/kaggle/working/results',

    # Data preprocessing
    'MAX_POSTS':      50,
    'MAX_WORDS':      70,

    # RAG
    'RAG_TOP_K':      5,
    'EMBED_MODEL':    'all-MiniLM-L6-v2',
    'EMBED_DIM':      384,

    # Agent
    'LLM_MODEL':      'Qwen/Qwen2.5-7B-Instruct',
    'MAX_AGENT_ITERATIONS': 5,          # ← was 8: most samples resolve in 3-4 steps

    # Long-term Memory
    'MEMORY_CONF_THRESHOLD': 0.80,

    # ── NLI Post-Verification Guardrail ──
    # Set to False to disable the guardrail entirely (no verification,
    # no rejection logic). Useful for ablation experiments.
    'GUARDRAIL_ENABLED':        True,

    # NLI Post-Verification: uses DeBERTa-v3-base (cross-encoder, NLI fine-tuned)
    # to independently verify the agent's predictions against the posts.
    # For each dimension, checks if the posts entail the predicted trait hypothesis.
    'NLI_MODEL':                 'cross-encoder/nli-deberta-v3-base',
    'GUARDRAIL_THRESHOLD':       0.50,   # min P(entailment) to accept
    'GUARDRAIL_PREMISE_MAX_CHARS': 800,  # max chars from posts for NLI premise
    'MAX_GUARDRAIL_RETRIES':     2,      # max rejections before forced accept
}

os.makedirs(CONFIG['RESULT_DIR'], exist_ok=True)

# ── GPU detection ──
NUM_GPUS = torch.cuda.device_count()
DEVICE   = torch.device('cuda' if NUM_GPUS > 0 else 'cpu')
print(f'🖥  GPUs available: {NUM_GPUS}  |  Device: {DEVICE}')
print(f'🛡️  Guardrail:     {"ON — NLI Post-Verification (DeBERTa)" if CONFIG["GUARDRAIL_ENABLED"] else "OFF — no verification"}')

# ── Constants ──
MBTI_TYPES  = ['infj','infp','intj','intp','isfj','isfp','istj','istp',
               'enfj','enfp','entj','entp','esfj','esfp','estj','estp']
MBTI_EXTRA  = ['introvert','extrovert','introverted','extroverted',
               'sensing','intuition','thinking','feeling','judging','perceiving']
ALL_MASK    = MBTI_TYPES + MBTI_EXTRA
MASK_RE     = re.compile(r'\b(' + '|'.join(ALL_MASK) + r')\b', re.IGNORECASE)

LABEL_COL   = 'type'
TEXT_COL    = 'posts'
LABEL_COLS  = ['label_ie', 'label_sn', 'label_tf', 'label_jp']
DIM_NAMES   = ['I/E', 'S/N', 'T/F', 'J/P']

## 📂 Cell 4 — Load & Preprocess Dataset

In [ ]:
# ── Load dataset (prefer 500K, fall back to 8K) ──
def load_dataset(cfg):
    for path in [cfg['DATA_PATH_500'], cfg['DATA_PATH_8K']]:
        try:
            df = pd.read_csv(path)
            print(f'✅ Loaded: {path}  shape={df.shape}')
            return df
        except FileNotFoundError:
            continue
    raise FileNotFoundError('❌ No dataset found. Add data and check paths.')

df_raw = load_dataset(CONFIG)


# ── Step 1: Map 16-class MBTI → 4 binary labels ──
def map_mbti_to_binary(mbti_str):
    m = mbti_str.strip().upper()
    return int(m[0]=='I'), int(m[1]=='N'), int(m[2]=='F'), int(m[3]=='J')

df_raw[['label_ie','label_sn','label_tf','label_jp']] = (
    df_raw[LABEL_COL].apply(lambda x: pd.Series(map_mbti_to_binary(x)))
)


# ── Step 2: Psycholinguistic masking + truncation ──
def preprocess_user_posts(raw_str, max_posts=CONFIG['MAX_POSTS'], max_words=CONFIG['MAX_WORDS']):
    posts = raw_str.split('|||')
    result = []
    for p in posts[:max_posts]:
        p = MASK_RE.sub('<type>', p.strip())
        p = ' '.join(p.split()[:max_words])
        if p:
            result.append(p)
    return result

print('⏳ Preprocessing posts (masking + truncation)...')
df_raw['processed_posts'] = df_raw[TEXT_COL].apply(preprocess_user_posts)
df_raw['concat_posts'] = df_raw['processed_posts'].apply(lambda x: ' ||| '.join(x))
print(f'✅ Preprocessing done. Rows: {len(df_raw)}')


# ── Step 3: Stratified split across ALL 4 axes jointly ──
df_raw['combo_label'] = df_raw[LABEL_COLS].astype(str).agg(''.join, axis=1)

train_df, temp_df = train_test_split(
    df_raw, test_size=0.4, stratify=df_raw['combo_label'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['combo_label'], random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'\n📊 Split → Train:{len(train_df)} | Val:{len(val_df)} | Test:{len(test_df)}')

## 📚 Cell 5 — MBTI Knowledge Base

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  MBTI KNOWLEDGE BASE (Domain Knowledge for RAG)
# ═══════════════════════════════════════════════════════════════

MBTI_DIM_KNOWLEDGE = {
    "I/E": (
        "Introversion (I) indicators in text: reflective and introspective language, longer and more "
        "complex sentences, frequent use of 'I think', 'in my opinion', hedging words like 'perhaps' "
        "or 'maybe', discussion of inner thoughts and ideas, fewer social references, preference for "
        "deep topics over small talk, lower frequency of exclamation marks, references to solitude and "
        "personal space, self-referential language, analytical introspection.\n"
        "Extraversion (E) indicators in text: action-oriented and energetic language, shorter direct "
        "sentences, frequent exclamation marks, references to social activities and groups, use of "
        "'we/us/our', discussion of events and people, more casual and conversational tone, references "
        "to parties, gatherings, collaboration, social energy, group dynamics."
    ),
    "S/N": (
        "Sensing (S) indicators in text: concrete and specific language, references to tangible details "
        "and facts, step-by-step descriptions, present-focused or past-experience based writing, "
        "practical vocabulary, sensory words (see, hear, feel, touch), specific numbers and dates, "
        "discussion of real-world events, hands-on experiences, routine references.\n"
        "Intuition (N) indicators in text: abstract and theoretical language, metaphors and analogies, "
        "discussion of possibilities and 'what if' scenarios, future-oriented thinking, conceptual "
        "vocabulary, references to patterns and connections, philosophical discussions, creative and "
        "imaginative expressions, symbolic language, big-picture focus."
    ),
    "T/F": (
        "Thinking (T) indicators in text: logical and analytical language, cause-effect reasoning "
        "using words like 'because', 'therefore', 'logically', objective and impersonal tone, "
        "critique and debate style, fewer emotional expressions, systematic analysis, focus on truth "
        "and accuracy, direct disagreement, technical vocabulary, problem-solving focus.\n"
        "Feeling (F) indicators in text: emotional and empathetic language, personal values expression, "
        "use of feeling words ('feel', 'love', 'care', 'heart'), harmony-seeking communication, "
        "people-focused discussions, supportive and encouraging tone, references to personal impact, "
        "moral judgments based on values rather than logic, relationship-oriented content."
    ),
    "J/P": (
        "Judging (J) indicators in text: decisive and definitive language using 'should', 'must', "
        "'will', structured and organized writing, closure-seeking statements, references to plans, "
        "schedules, and goals, preference for order, list-making tendency, strong opinions stated as "
        "facts, completion-focused language, methodical approach.\n"
        "Perceiving (P) indicators in text: open-ended and exploratory language using 'maybe', "
        "'might', 'what about', flexible and adaptive writing, multiple options considered, "
        "spontaneous topic changes, casual attitude toward deadlines, curiosity-driven exploration, "
        "'going with the flow' references, incomplete thoughts, tangential thinking."
    ),
}

MBTI_TYPE_PROFILES = {
    "INTJ": "Ni-Te-Fi-Se. Strategic, independent, determined. Precise analytical language, long-term vision, minimal small talk, direct communication, confident assertions, systems thinking, rare emotional expression, future-planning focus.",
    "INTP": "Ti-Ne-Si-Fe. Analytical, curious, innovative. Complex sentence structures, explores multiple angles, theoretical vocabulary, questions everything, logical precision, detached tone, idea-driven, loves intellectual debate.",
    "ENTJ": "Te-Ni-Se-Fi. Decisive, ambitious, commanding. Direct and authoritative language, efficiency-focused, goal-oriented, structured arguments, leadership references, action-oriented planning, strategic communication.",
    "ENTP": "Ne-Ti-Fe-Si. Quick-witted, argumentative, inventive. Playful language, devil's advocate positions, humor and sarcasm, idea-jumping, debate-seeking, creative connections, challenges assumptions, energetic discourse.",
    "INFJ": "Ni-Fe-Ti-Se. Insightful, principled, compassionate. Metaphorical language, deep reflections, idealistic expressions, future-oriented, empathetic but private, symbolic thinking, meaningful connections, visionary language.",
    "INFP": "Fi-Ne-Si-Te. Idealistic, empathetic, creative. Poetic and personal language, value-driven expressions, emotional depth, authenticity focus, imaginative descriptions, identity exploration, artistic sensibility.",
    "ENFJ": "Fe-Ni-Se-Ti. Charismatic, inspiring, empathetic. Encouraging and motivational language, people-focused, visionary statements, warm tone, community references, natural leadership, harmony-building communication.",
    "ENFP": "Ne-Fi-Te-Si. Enthusiastic, creative, sociable. Energetic language, many exclamations, rapid topic shifts, passionate expressions, optimistic tone, brainstorming style, explores possibilities, emotionally expressive.",
    "ISTJ": "Si-Te-Fi-Ne. Practical, reliable, methodical. Structured factual language, duty references, traditional values, step-by-step descriptions, concrete details, rule-following, historical references.",
    "ISFJ": "Si-Fe-Ti-Ne. Caring, reliable, observant. Supportive language, detail-oriented descriptions, tradition-respecting, nurturing tone, practical helpfulness, loyalty-focused, modest expression.",
    "ESTJ": "Te-Si-Ne-Fi. Organized, direct, traditional. Clear directives, fact-based arguments, efficiency-focused, authoritative tone, rule-following references, practical leadership.",
    "ESFJ": "Fe-Si-Ne-Ti. Caring, sociable, traditional. Warm and inclusive language, social harmony focus, community references, supportive expressions, practical care, people-oriented.",
    "ISTP": "Ti-Se-Ni-Fe. Practical, observant, analytical. Concise factual language, action-focused, problem-solving oriented, minimal emotional expression, hands-on references, independent thinking.",
    "ISFP": "Fi-Se-Ni-Te. Gentle, sensitive, artistic. Aesthetic language, present-moment focus, personal values, sensory descriptions, quiet emotional depth, creative expression.",
    "ESTP": "Se-Ti-Fe-Ni. Bold, direct, perceptive. Action-oriented short sentences, present-focused, practical and energetic, humor, risk-taking references, competitive language.",
    "ESFP": "Se-Fi-Te-Ni. Spontaneous, energetic, playful. Vivid sensory language, social engagement, fun-focused, enthusiastic tone, living-in-the-moment expressions, entertaining style.",
}

print(f'✅ Knowledge base ready: {len(MBTI_DIM_KNOWLEDGE)} dimension descriptions + {len(MBTI_TYPE_PROFILES)} type profiles')

## 🔍 Cell 6 — Build LangChain FAISS VectorStores

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  BUILD LANGCHAIN FAISS VECTORSTORES
# ═══════════════════════════════════════════════════════════════

# ── LangChain Embeddings (wraps sentence-transformers) ──
print('⏳ Loading LangChain HuggingFaceEmbeddings...')
lc_embeddings = HuggingFaceEmbeddings(
    model_name=CONFIG['EMBED_MODEL'],
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

# ── VectorStore 1: Training posts → similar-post retrieval (few-shot) ──
print('⏳ Building LangChain FAISS vectorstore for training posts...')
post_documents = [
    Document(
        page_content=row['concat_posts'],
        metadata={'type': row[LABEL_COL], 'index': i}
    )
    for i, row in train_df.iterrows()
]

# Build in batches to avoid memory issues
FAISS_BATCH = 512
post_vectorstore = LangChainFAISS.from_documents(post_documents[:FAISS_BATCH], lc_embeddings)
for start in range(FAISS_BATCH, len(post_documents), FAISS_BATCH):
    batch = post_documents[start:start + FAISS_BATCH]
    post_vectorstore.add_documents(batch)
    if (start // FAISS_BATCH) % 20 == 0:
        print(f'  ... {start + len(batch)}/{len(post_documents)} posts indexed')

post_retriever = post_vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': CONFIG['RAG_TOP_K']}
)
print(f'✅ Post vectorstore: {len(post_documents)} documents → LangChain FAISS retriever')


# ── VectorStore 2: MBTI Knowledge Base ──
print('⏳ Building LangChain FAISS vectorstore for MBTI knowledge...')
knowledge_documents = []
for dim_name, text in MBTI_DIM_KNOWLEDGE.items():
    knowledge_documents.append(Document(
        page_content=text,
        metadata={'label': dim_name, 'doc_type': 'dimension'}
    ))
for type_name, profile in MBTI_TYPE_PROFILES.items():
    knowledge_documents.append(Document(
        page_content=profile,
        metadata={'label': type_name, 'doc_type': 'profile'}
    ))

knowledge_vectorstore = LangChainFAISS.from_documents(knowledge_documents, lc_embeddings)
knowledge_retriever = knowledge_vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 4}
)
print(f'✅ Knowledge vectorstore: {len(knowledge_documents)} chunks → LangChain FAISS retriever')


# ── Pre-embed test posts (use underlying model for speed) ──
print('⏳ Encoding test posts...')
from sentence_transformers import SentenceTransformer
_st_model = SentenceTransformer(CONFIG['EMBED_MODEL'], device='cpu')
test_texts_for_rag = test_df['concat_posts'].tolist()
test_embeddings_np = _st_model.encode(
    test_texts_for_rag, batch_size=128, show_progress_bar=True,
    normalize_embeddings=True
).astype(np.float32)
print(f'✅ Test embeddings ready: {test_embeddings_np.shape}')

## 🧠 Cell 7 — Load LLM via LangChain HuggingFacePipeline

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  LOAD LLM via LangChain HuggingFacePipeline
# ═══════════════════════════════════════════════════════════════

# ── Pre-flight CUDA check ──
_torch_cuda = torch.__version__
print(f'🔍 torch={_torch_cuda}, CUDA={torch.version.cuda}')
if 'cu130' in _torch_cuda or 'cu131' in _torch_cuda:
    raise RuntimeError(
        f'torch {_torch_cuda} needs CUDA 13 but Kaggle has CUDA 12. '
        f'Restart kernel (Runtime > Restart session) and re-run all cells.'
    )

LLM_MODEL_NAME = CONFIG['LLM_MODEL']

print(f'⏳ Loading {LLM_MODEL_NAME} (4-bit quantization)...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME, trust_remote_code=True)
llm_model_raw = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

hf_pipe = hf_pipeline(
    'text-generation',
    model=llm_model_raw,
    tokenizer=llm_tokenizer,
    max_new_tokens=150,             # ← was 256: ReAct steps are short
    do_sample=False,                # ← greedy decoding = faster, deterministic
    return_full_text=False,
)

# ── Wrap as LangChain LLM ──
# NOTE: create_react_agent adds stop=["\nObservation:"] by default,
# which HuggingFacePipeline handles via post-processing (strips text at stop token).
# Combined with RobustReActParser (Cell 8), this prevents hallucinated tool outputs.
llm = HuggingFacePipeline(pipeline=hf_pipe)

print(f'✅ LLM loaded: {LLM_MODEL_NAME}  |  dtype=4bit-nf4  |  max_new_tokens=150')
print(f'   LangChain wrapper: HuggingFacePipeline')
print(f'   Decoding: greedy (do_sample=False) — faster than sampling')
print(f'   ReAct stop: LangChain built-in stop=[\\nObservation:] + RobustReActParser')

## 🛡️ Cell 7b — NLI Post-Verification Guardrail (Independent 3rd-Party Evaluator)

**NLI Post-Verification Guardrail** — Uses an **independent** DeBERTa-v3-base model (cross-encoder, NLI fine-tuned) to verify the agent's predictions. The evaluator is completely separate from the LLM reasoning pipeline.

| Role | Model | Purpose |
|------|-------|---------|
| **Generator** | Qwen2.5-7B | Produces MBTI prediction + reasoning via ReAct agent |
| **Verifier** | DeBERTa-v3-base (~184M) | Independently checks if posts entail each dimension's claim |

**How it works (Hậu kiểm — Post-Verification):**
1. Agent predicts MBTI type via ReAct loop (analyze → retrieve → predict)
2. From the prediction, construct NLI hypothesis per dimension (from `EVAL_HYPOTHESES`)
3. NLI model checks: **premise** (posts) → **hypothesis** (predicted trait claim)
4. Returns P(entailment), P(neutral), P(contradiction) per dimension
5. Reject when P(entailment) < `GUARDRAIL_THRESHOLD` → agent must revise
6. After `MAX_GUARDRAIL_RETRIES` rejections, force-accept to prevent infinite loops

**Why independent NLI is better than LLM self-evaluation:**
- **Independent** — separate model avoids self-evaluation bias (LLM grading its own work)
- **Fast** — DeBERTa cross-encoder ~50ms for 4 pairs (vs seconds for LLM inference)
- **Calibrated** — NLI models produce well-calibrated entailment probabilities
- **No claim extraction needed** — structured `EVAL_HYPOTHESES` provide the claims directly

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  NLI POST-VERIFICATION GUARDRAIL (Independent 3rd-Party)
# ═══════════════════════════════════════════════════════════════
# Uses DeBERTa-v3-base (cross-encoder, NLI fine-tuned) — a completely
# independent model from the LLM — to verify the agent's predictions.
#
# For each MBTI dimension, constructs (premise, hypothesis):
#   premise    = user's posts (truncated to GUARDRAIL_PREMISE_MAX_CHARS)
#   hypothesis = claim about the predicted trait (from EVAL_HYPOTHESES)
# and checks P(entailment) using the NLI cross-encoder.
#
# Controlled by CONFIG['GUARDRAIL_ENABLED']:
#   True  → run NLI verification, reject predictions with low entailment
#   False → skip verification entirely
# ═══════════════════════════════════════════════════════════════

from sentence_transformers import CrossEncoder

# ── Hypothesis Templates (per MBTI dimension × letter) ──
EVAL_HYPOTHESES = {
    'ie': {
        'I': "The author prefers solitude, reflects on inner thoughts, and avoids social activities.",
        'E': "The author enjoys social activities, group interactions, and energetic conversations.",
    },
    'sn': {
        'S': "The author focuses on concrete facts, practical details, and real-world experiences.",
        'N': "The author focuses on abstract ideas, theories, possibilities, and imagination.",
    },
    'tf': {
        'T': "The author reasons logically, analyzes objectively, and values truth over feelings.",
        'F': "The author expresses emotions, values harmony, and cares about people's feelings.",
    },
    'jp': {
        'J': "The author prefers planning, structure, schedules, and decisive conclusions.",
        'P': "The author prefers flexibility, spontaneity, open options, and going with the flow.",
    },
}

_OPPOSITE = {'I': 'E', 'E': 'I', 'S': 'N', 'N': 'S', 'T': 'F', 'F': 'T', 'J': 'P', 'P': 'J'}

# ── Load NLI Model (independent 3rd-party evaluator) ──
if CONFIG['GUARDRAIL_ENABLED']:
    print(f'⏳ Loading NLI model: {CONFIG["NLI_MODEL"]}...')
    nli_model = CrossEncoder(CONFIG['NLI_MODEL'], max_length=512, device='cpu')

    # Get label indices from model config (robust to different label orderings)
    _id2label = nli_model.model.config.id2label
    _NLI_ENTAIL_IDX  = [int(k) for k, v in _id2label.items() if 'entail' in v.lower()][0]
    _NLI_CONTRA_IDX  = [int(k) for k, v in _id2label.items() if 'contra' in v.lower()][0]
    _NLI_NEUTRAL_IDX = [int(k) for k, v in _id2label.items() if 'neutral' in v.lower()][0]

    _nli_params = sum(p.numel() for p in nli_model.model.parameters()) / 1e6
    print(f'🛡️ NLI Post-Verification Guardrail: ON')
    print(f'   Model: {CONFIG["NLI_MODEL"]} ({_nli_params:.0f}M params)')
    print(f'   Labels: {_id2label}')
    print(f'   Threshold: P(entailment) ≥ {CONFIG["GUARDRAIL_THRESHOLD"]}')
    print(f'   Max retries: {CONFIG["MAX_GUARDRAIL_RETRIES"]}')
    print(f'   ⚡ ~50ms for 4 dimension pairs (independent from LLM)')
else:
    nli_model = None
    print('⏭️  Guardrail DISABLED')
    print('   Set CONFIG["GUARDRAIL_ENABLED"] = True to enable')


def guardrail_score_prediction(posts_text, predicted_dims, detailed=False):
    """Verify each predicted MBTI dimension using NLI (DeBERTa cross-encoder).

    For each dimension, checks if the posts (premise) entail the predicted
    trait hypothesis. Returns P(entailment) per dimension as scores in [0.0, 1.0].

    This is a completely independent verification — the NLI model has never
    seen the LLM's reasoning, only the raw posts and the predicted claim.

    Returns {} (or ({}, {})) when guardrail is disabled.
    """
    if not CONFIG['GUARDRAIL_ENABLED'] or nli_model is None:
        return ({}, {}) if detailed else {}

    premise = posts_text[:CONFIG['GUARDRAIL_PREMISE_MAX_CHARS']]
    dim_keys = ['ie', 'sn', 'tf', 'jp']

    # Construct NLI pairs: (premise, hypothesis) for each dimension
    pairs = []
    for dk in dim_keys:
        letter = predicted_dims.get(dk, '?').upper()
        hyp = EVAL_HYPOTHESES.get(dk, {}).get(letter, 'Unknown personality trait.')
        pairs.append((premise, hyp))

    try:
        # CrossEncoder returns softmax probabilities: shape (4, 3)
        probs = nli_model.predict(pairs, apply_softmax=True)
    except Exception:
        scores = {k: 0.5 for k in dim_keys}
        if detailed:
            return scores, {k: {'predicted': predicted_dims.get(dk, '?').upper(),
                                'score': 0.5, 'label': 'ERROR'} for k in dim_keys}
        return scores

    scores = {}
    details = {}
    for i, dk in enumerate(dim_keys):
        p_entail  = float(probs[i, _NLI_ENTAIL_IDX])
        p_contra  = float(probs[i, _NLI_CONTRA_IDX])
        p_neutral = float(probs[i, _NLI_NEUTRAL_IDX])
        scores[dk] = p_entail

        if detailed:
            letter = predicted_dims.get(dk, '?').upper()
            label = ('ENTAILMENT' if p_entail >= p_contra and p_entail >= p_neutral
                     else 'CONTRADICTION' if p_contra >= p_neutral
                     else 'NEUTRAL')
            details[dk] = {
                'predicted': letter,
                'opposite': _OPPOSITE.get(letter, '?'),
                'score': p_entail,
                'p_entail': p_entail,
                'p_neutral': p_neutral,
                'p_contra': p_contra,
                'label': label,
            }

    if detailed:
        return scores, details
    return scores


# ── Sanity checks ──
if CONFIG['GUARDRAIL_ENABLED']:
    _test1 = "I love going to parties and meeting new people! We always have fun together as a group!"
    _s1, _d1 = guardrail_score_prediction(_test1, {'ie': 'E', 'sn': 'N', 'tf': 'F', 'jp': 'P'}, detailed=True)
    print(f'\n   Sanity check 1 — extraverted text, predict E:')
    for dk in ['ie', 'sn', 'tf', 'jp']:
        d = _d1[dk]
        print(f'     {dk}: P(ent)={d["p_entail"]:.3f}  P(neu)={d["p_neutral"]:.3f}  '
              f'P(con)={d["p_contra"]:.3f}  → {d["label"]}')

    _test2 = "I prefer spending time alone reading books. I think deeply about life and rarely go out."
    _s2, _d2 = guardrail_score_prediction(_test2, {'ie': 'I', 'sn': 'N', 'tf': 'T', 'jp': 'J'}, detailed=True)
    print(f'\n   Sanity check 2 — introverted text, predict I:')
    for dk in ['ie', 'sn', 'tf', 'jp']:
        d = _d2[dk]
        print(f'     {dk}: P(ent)={d["p_entail"]:.3f}  P(neu)={d["p_neutral"]:.3f}  '
              f'P(con)={d["p_contra"]:.3f}  → {d["label"]}')

    _s3, _d3 = guardrail_score_prediction(_test2, {'ie': 'E', 'sn': 'N', 'tf': 'T', 'jp': 'J'}, detailed=True)
    print(f'\n   Sanity check 3 — introverted text, predict E (WRONG — expect low entailment):')
    for dk in ['ie', 'sn', 'tf', 'jp']:
        d = _d3[dk]
        print(f'     {dk}: P(ent)={d["p_entail"]:.3f}  P(neu)={d["p_neutral"]:.3f}  '
              f'P(con)={d["p_contra"]:.3f}  → {d["label"]}')

    print(f'\n   Threshold: P(entailment) ≥ {CONFIG["GUARDRAIL_THRESHOLD"]}')

## 🔧 Cell 8 — LangChain Tools + ReAct Agent

**LangChain Components:**
1. `@tool` decorated functions — automatic JSON schema generation
2. `create_react_agent()` — ReAct reasoning loop
3. `AgentExecutor` — execution engine with iteration control

**Tools:**
1. `analyze_text` — Extract psycholinguistic features
2. `retrieve_similar_posts` — LangChain FAISS retriever for few-shot examples
3. `retrieve_mbti_knowledge` — LangChain FAISS retriever for domain knowledge (LLM specifies query)
4. `recall_experience` — Query long-term memory store
5. `submit_prediction` — Submit final MBTI prediction (**with NLI Post-Verification Guardrail**)

**Guardrail (NLI Post-Verification — Independent 3rd Party):**
- `submit_prediction` runs predictions through the **independent DeBERTa-v3-base NLI model**
- Constructs (premise=posts, hypothesis=predicted trait) → checks P(entailment)
- If P(entailment) < `GUARDRAIL_THRESHOLD` for any dimension → **REJECTED**
- After `MAX_GUARDRAIL_RETRIES` (2) rejections, the prediction is force-accepted to prevent infinite loops
- The NLI model is completely separate from the LLM — no self-evaluation bias

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  LANGCHAIN TOOLS (via @tool decorator)
# ═══════════════════════════════════════════════════════════════

# ── Shared state: set per-sample before agent.invoke() ──
_current_posts_text = ""
_current_query_embedding = None

# ── Long-term Experience Memory ──
_experience_store = []
_experience_faiss_index = None
_total_memories_stored = 0
_pattern_memory = {}

# ── Guardrail state (reset per sample) ──
_guardrail_retries = 0


@tool
def analyze_text(dummy: str = "") -> str:
    """Extract psycholinguistic features from the current posts:
    word count, avg word/sentence length, question ratio, pronoun percentages,
    emotion/thinking/abstract word percentages.
    Call this first to get statistical signals about personality.
    Input: any string (ignored, operates on current posts)."""
    posts_text = _current_posts_text
    words = posts_text.lower().split()
    total = len(words)
    if total == 0:
        return json.dumps({})
    sentences = max(posts_text.count('.') + posts_text.count('?') + posts_text.count('!'), 1)
    features = {
        'total_words': total,
        'avg_word_len': round(np.mean([len(w) for w in words]), 2),
        'avg_sent_len': round(total / sentences, 1),
        'question_ratio': round(posts_text.count('?') / sentences, 3),
        'exclam_ratio': round(posts_text.count('!') / sentences, 3),
        'first_person_pct': round(
            sum(1 for w in words if w in {'i','me','my','myself','mine'}) / total * 100, 2),
        'we_pct': round(
            sum(1 for w in words if w in {'we','us','our','ourselves'}) / total * 100, 2),
        'emotion_pct': round(
            sum(1 for w in words if w in {
                'feel','feeling','love','hate','happy','sad','angry','beautiful',
                'wonderful','terrible','amazing','awful','care','heart','soul',
                'passion','joy','fear','hope','wish'
            }) / total * 100, 2),
        'thinking_pct': round(
            sum(1 for w in words if w in {
                'think','thought','logic','logical','reason','because','therefore',
                'analyze','analysis','system','theory','hypothesis','evidence','data','fact'
            }) / total * 100, 2),
        'abstract_pct': round(
            sum(1 for w in words if w in {
                'concept','idea','theory','possibility','imagine','philosophy',
                'meaning','purpose','pattern','connection','potential','vision',
                'insight','metaphor','symbol','abstract','infinite'
            }) / total * 100, 2),
        'url_count': posts_text.lower().count('http'),
    }
    return json.dumps(features, indent=2)


@tool
def retrieve_similar_posts(num_posts: str = "5") -> str:
    """Find the most similar labeled posts from the training set as few-shot examples.
    Args:
        num_posts: Number of similar posts to retrieve (1-10). Default: 5."""
    try:
        top_k = int(re.search(r'\d+', str(num_posts)).group())
    except (AttributeError, ValueError):
        top_k = 5
    top_k = min(max(1, top_k), 10)
    posts_text = _current_posts_text
    docs = post_vectorstore.similarity_search_with_score(posts_text[:1000], k=top_k)
    lines = []
    for i, (doc, score) in enumerate(docs):
        mbti = doc.metadata.get('type', '?')
        preview = doc.page_content[:200]
        lines.append(f"  {i+1}. [{mbti}] (sim={score:.3f}): {preview}")
    return f"Top {len(docs)} similar posts:\n" + '\n'.join(lines)


@tool
def retrieve_mbti_knowledge(query: str) -> str:
    """Search the MBTI knowledge base for dimension descriptions and type profiles.
    Use a TARGETED query about the dimension(s) you are uncertain about.
    Args:
        query: Search query, e.g. 'thinking vs feeling analytical emotional language'"""
    docs = knowledge_vectorstore.similarity_search(query, k=4)
    lines = []
    for doc in docs:
        label = doc.metadata.get('label', '?')
        lines.append(f'  [{label}]: {doc.page_content[:300]}')
    return f"{len(docs)} knowledge chunks:\n" + '\n'.join(lines)


@tool
def recall_experience(dummy: str = "") -> str:
    """Query long-term memory for similar posts previously classified with high confidence.
    Input: any string (ignored, uses current post embedding)."""
    global _experience_faiss_index
    if _experience_faiss_index is None or _experience_faiss_index.ntotal == 0:
        return f"No relevant past experiences (memory_size={len(_experience_store)})"
    query_emb = _current_query_embedding
    k = min(2, _experience_faiss_index.ntotal)
    scores, indices = _experience_faiss_index.search(query_emb.reshape(1, -1), k)
    results = []
    for idx, score in zip(indices[0], scores[0]):
        if idx >= 0 and score > 0.5:
            exp = _experience_store[idx]
            results.append(
                f"  [{exp['type']}] (sim={float(score):.2f}, "
                f"conf={exp['avg_conf']:.2f}): {exp['reasoning'][:200]}"
            )
    if not results:
        return f"No similar past experiences found (memory_size={len(_experience_store)})"
    return f"{len(results)} past experiences:\n" + '\n'.join(results)


@tool
def submit_prediction(prediction: str) -> str:
    """Submit MBTI prediction. Format: XXXX ie_conf sn_conf tf_conf jp_conf
    Example: INTP 0.85 0.75 0.90 0.80
    The 4-letter MBTI type, then 4 confidence values (0.5-1.0) for I/E, S/N, T/F, J/P.
    Call ONLY after gathering evidence from at least 2 other tools.
    WARNING: An independent NLI model (DeBERTa) will verify your prediction against the posts.
    If the entailment score is too low, your prediction will be REJECTED.
    Args:
        prediction: MBTI type and confidences, e.g. 'INTP 0.85 0.75 0.90 0.80'"""
    global _guardrail_retries

    # ── Robust parsing ──
    clean = prediction.replace(',', ' ').replace(';', ' ')
    type_match = re.search(r'[IiEe][SsNn][TtFf][JjPp]', clean)
    mbti_type = type_match.group().upper() if type_match else 'INFP'

    numbers = []
    for x in re.findall(r'\b[01]\.\d+\b', clean):
        numbers.append(float(x))

    ie_conf = numbers[0] if len(numbers) > 0 else 0.5
    sn_conf = numbers[1] if len(numbers) > 1 else 0.5
    tf_conf = numbers[2] if len(numbers) > 2 else 0.5
    jp_conf = numbers[3] if len(numbers) > 3 else 0.5

    result = {
        'type': mbti_type,
        'ie': mbti_type[0], 'ie_conf': max(0.0, min(1.0, ie_conf)),
        'sn': mbti_type[1], 'sn_conf': max(0.0, min(1.0, sn_conf)),
        'tf': mbti_type[2], 'tf_conf': max(0.0, min(1.0, tf_conf)),
        'jp': mbti_type[3], 'jp_conf': max(0.0, min(1.0, jp_conf)),
    }

    # ══════════════════════════════════════════════════════════
    #  NLI POST-VERIFICATION GUARDRAIL (DeBERTa-v3-base)
    #  Independent 3rd-party — completely separate from LLM
    #  Skipped when CONFIG['GUARDRAIL_ENABLED'] = False
    # ══════════════════════════════════════════════════════════
    if CONFIG['GUARDRAIL_ENABLED'] and _guardrail_retries < CONFIG['MAX_GUARDRAIL_RETRIES']:
        guard_threshold = CONFIG['GUARDRAIL_THRESHOLD']
        predicted_dims = {
            'ie': result['ie'], 'sn': result['sn'],
            'tf': result['tf'], 'jp': result['jp'],
        }
        guard_scores = guardrail_score_prediction(_current_posts_text, predicted_dims)

        dim_names_map = {'ie': 'I/E', 'sn': 'S/N', 'tf': 'T/F', 'jp': 'J/P'}
        weak_dims = [
            (dim_names_map[k], guard_scores.get(k, 0.0))
            for k in ['ie', 'sn', 'tf', 'jp']
            if guard_scores.get(k, 0.0) < guard_threshold
        ]

        if weak_dims:
            _guardrail_retries += 1
            weak_str = ', '.join(
                f'{name}={score:.4f}' for name, score in weak_dims
            )
            for k in ['ie', 'sn', 'tf', 'jp']:
                result[f'{k}_guard'] = guard_scores.get(k, 0.0)
            return (
                f"PREDICTION REJECTED by NLI Verifier "
                f"(attempt {_guardrail_retries}/{CONFIG['MAX_GUARDRAIL_RETRIES']}): "
                f"P(entailment) too low on [{weak_str}] "
                f"(threshold={guard_threshold}). "
                f"The independent NLI model does not find sufficient textual evidence. "
                f"Call retrieve_mbti_knowledge with a query about the weak dimension(s), "
                f"re-analyze the evidence, then call submit_prediction with a REVISED prediction."
            )

        # Guardrail passed — store scores as metadata
        for k in ['ie', 'sn', 'tf', 'jp']:
            result[f'{k}_guard'] = guard_scores.get(k, 0.0)

    # ── Accept prediction (guardrail passed, off, or max retries reached) ──
    return "PREDICTION_SUBMITTED:" + json.dumps(result)


# ═══════════════════════════════════════════════════════════════
#  ROBUST OUTPUT PARSER (fixes LLM hallucination of Final Answer + Action)
# ═══════════════════════════════════════════════════════════════
from langchain.agents.output_parsers import ReActSingleInputOutputParser
from langchain_core.exceptions import OutputParserException

class RobustReActParser(ReActSingleInputOutputParser):
    """Custom parser that handles common 7B-model failure modes:
    1. LLM generates BOTH 'Action:' and 'Final Answer:' in one response
    2. LLM generates only 'Thought:' without 'Action:' (token limit hit)
    3. LLM hallucinates 'Observation:' blocks
    4. LLM repeats 'Final Answer:' lines
    """
    def parse(self, text: str):
        if "\nObservation:" in text:
            text = text.split("\nObservation:")[0]
        if "\nObservation :" in text:
            text = text.split("\nObservation :")[0]

        has_action = "Action:" in text and "Action Input:" in text
        has_final  = "Final Answer:" in text

        if has_action and has_final:
            action_idx = text.index("Action:")
            final_idx  = text.index("Final Answer:")
            if action_idx < final_idx:
                text = text[:final_idx].rstrip()
            else:
                text = text[:action_idx].rstrip()

        if has_final and not has_action:
            fa_idx = text.index("Final Answer:")
            answer = text[fa_idx + len("Final Answer:"):].strip()
            if "\nFinal Answer:" in answer:
                answer = answer.split("\nFinal Answer:")[0].strip()
            text = text[:fa_idx] + "Final Answer: " + answer

        if not has_action and not has_final:
            import re as _re
            mbti_match = _re.search(r'\b([IiEe][SsNn][TtFf][JjPp])\b', text)
            if mbti_match:
                mbti_type = mbti_match.group(1).upper()
                text = (f"Thought: Auto-extracted prediction from truncated thought.\n"
                        f"Action: submit_prediction\n"
                        f"Action Input: {mbti_type} 0.70 0.70 0.70 0.70")

        return super().parse(text)


# ═══════════════════════════════════════════════════════════════
#  LANGCHAIN ReAct AGENT
# ═══════════════════════════════════════════════════════════════

tools = [analyze_text, retrieve_similar_posts, retrieve_mbti_knowledge,
         recall_experience, submit_prediction]

# Build prompt — include guardrail warning only when enabled
_guardrail_prompt_section = ""
if CONFIG['GUARDRAIL_ENABLED']:
    _guardrail_prompt_section = """
IMPORTANT — NLI Guardrail (Independent Verifier):
An independent NLI model (DeBERTa) will verify whether your prediction is supported by the posts.
If any dimension lacks sufficient textual evidence (low entailment), your prediction will be REJECTED.
When rejected, reconsider the weak dimensions and revise your prediction."""

REACT_PROMPT = PromptTemplate.from_template(
"""You are an MBTI personality classifier. Analyze posts and classify the author's MBTI type.

MBTI dimensions: I/E (Introversion vs Extraversion), S/N (Sensing vs Intuition), T/F (Thinking vs Feeling), J/P (Judging vs Perceiving).

Tools available:
{tools}

Tool names: {tool_names}

FORMAT RULES (follow EXACTLY):
- Write Thought, then Action, then Action Input. STOP there. Do NOT write anything else after Action Input.
- Do NOT write "Observation:" — the system provides observations automatically.
- Do NOT write "Final Answer:" until you have called submit_prediction and seen PREDICTION_SUBMITTED in the observation.

Format for tool calls:
Thought: [reasoning]
Action: [tool name]
Action Input: [input]

Format for final answer (ONLY after seeing PREDICTION_SUBMITTED):
Final Answer: [the PREDICTION_SUBMITTED result]
""" + _guardrail_prompt_section + """
STEPS:
1. Action: analyze_text → get linguistic stats
2. Action: retrieve_similar_posts → Action Input: 5
3. Action: submit_prediction → Action Input: XXXX ie_conf sn_conf tf_conf jp_conf (e.g. INTP 0.85 0.75 0.90 0.80)
4. If rejected by evaluator, Action: retrieve_mbti_knowledge → query the weak dimension, then resubmit with a REVISED prediction

Posts to classify:
{input}

{agent_scratchpad}""")

react_agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=REACT_PROMPT,
    output_parser=RobustReActParser(),
)

print('✅ LangChain Agent created')
print(f'   Agent type:     create_react_agent (ReAct) + RobustReActParser')
print(f'   LLM:            HuggingFacePipeline ({LLM_MODEL_NAME})')
print(f'   Tools:          {[t.name for t in tools]}')
print(f'   Max iterations: {CONFIG["MAX_AGENT_ITERATIONS"]}')
if CONFIG['GUARDRAIL_ENABLED']:
    print(f'   Guardrail:      NLI Post-Verification ({CONFIG["NLI_MODEL"]}) — '
          f'reject if P(entailment) < {CONFIG["GUARDRAIL_THRESHOLD"]} '
          f'(max {CONFIG["MAX_GUARDRAIL_RETRIES"]} retries)')
else:
    print(f'   Guardrail:      OFF — predictions accepted without verification')
print(f'   VectorStores:   post_vectorstore ({len(post_documents)} docs) + '
      f'knowledge_vectorstore ({len(knowledge_documents)} docs)')
print(f'   Parser:         RobustReActParser (handles dual Action+FinalAnswer)')

## 🔬 Cell 8b — DEMO: Single User Predictions with Full Agent Trace

Run the LangChain agent on a small sample to visualize the autonomous ReAct loop — where the LLM reasons, selects tools, and generates dynamic arguments via `AgentExecutor`.

**Watch for NLI guardrail rejections** — when the independent DeBERTa NLI model determines P(entailment) is too low for a dimension, the prediction is rejected and the agent must revise.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════════

DIM_LETTER_MAP_INV = {
    'ie': {1: 'I', 0: 'E'},
    'sn': {1: 'N', 0: 'S'},
    'tf': {1: 'F', 0: 'T'},
    'jp': {1: 'J', 0: 'P'},
}


def store_experience(query_embedding, posts_text, result, trace_text=""):
    global _experience_faiss_index, _total_memories_stored
    confs = [result.get(f'{d}_conf', 0) for d in ['ie', 'sn', 'tf', 'jp']]
    min_conf = min(confs)
    avg_conf = float(np.mean(confs))
    if min_conf < CONFIG['MEMORY_CONF_THRESHOLD']:
        return False
    experience = {
        'type': result['type'],
        'posts_preview': posts_text[:300],
        'result': result,
        'reasoning': trace_text[:500] if trace_text else 'Direct classification',
        'avg_conf': avg_conf,
    }
    _experience_store.append(experience)
    emb = query_embedding.copy().reshape(1, -1).astype(np.float32)
    if _experience_faiss_index is None:
        _experience_faiss_index = faiss.IndexFlatIP(emb.shape[1])
    _experience_faiss_index.add(emb)
    _total_memories_stored += 1
    return True


def parse_agent_result(output_text):
    m = re.search(r'PREDICTION_SUBMITTED:\s*(\{[^}]+\})', output_text)
    if m:
        try:
            return json.loads(m.group(1))
        except (json.JSONDecodeError, ValueError):
            pass
    for match in re.finditer(r'\{[^{}]+\}', output_text):
        try:
            obj = json.loads(match.group())
            if 'type' in obj or 'ie' in obj:
                return validate_result(obj)
        except (json.JSONDecodeError, ValueError):
            continue
    return None


def validate_result(obj):
    t = obj.get('type', obj.get('mbti_type', '')).upper()
    for dim, idx, pair in [('ie', 0, 'IE'), ('sn', 1, 'SN'),
                            ('tf', 2, 'TF'), ('jp', 3, 'JP')]:
        letter = obj.get(dim, '').upper()
        if letter not in pair:
            letter = (t[idx].upper()
                      if len(t) > idx and t[idx].upper() in pair
                      else pair[0])
        obj[dim] = letter
        conf = obj.get(f'{dim}_conf', 0.5)
        try:
            conf = max(0.0, min(1.0, float(conf)))
        except (ValueError, TypeError):
            conf = 0.5
        obj[f'{dim}_conf'] = conf
    obj['type'] = (obj['ie'] + obj['sn'] + obj['tf'] + obj['jp']).upper()
    return obj


def run_agent_on_sample(posts_text, query_embedding, verbose=False):
    """Run LangChain AgentExecutor on a single sample."""
    global _current_posts_text, _current_query_embedding, _guardrail_retries
    _current_posts_text = posts_text
    _current_query_embedding = query_embedding
    _guardrail_retries = 0

    executor = AgentExecutor(
        agent=react_agent,
        tools=tools,
        max_iterations=CONFIG['MAX_AGENT_ITERATIONS'],
        verbose=verbose,
        handle_parsing_errors=True,
        return_intermediate_steps=True,
    )

    try:
        result = executor.invoke({'input': posts_text[:1200]})
        output = result.get('output', '')
        steps = result.get('intermediate_steps', [])

        parsed = parse_agent_result(output)
        if parsed is None:
            for action, observation in steps:
                if hasattr(action, 'tool') and action.tool == 'submit_prediction':
                    parsed = parse_agent_result(str(observation))
                    if parsed:
                        break
        if parsed:
            parsed = validate_result(parsed)
        return parsed, output, steps
    except Exception as e:
        return None, f"Error: {str(e)}", []


def fallback_predict(posts_text, query_embedding):
    """Direct LLM prediction when agent loop fails — still uses RAG context."""
    similar_docs = post_vectorstore.similarity_search_with_score(posts_text[:1000], k=5)
    knowledge_docs = knowledge_vectorstore.similarity_search(
        "MBTI personality dimensions introversion extraversion", k=4
    )
    sim_str = '\n'.join(
        f"  [{d.metadata.get('type','?')}] (sim={s:.2f}): {d.page_content[:200]}"
        for d, s in similar_docs
    )
    know_str = '\n'.join(
        f"  [{d.metadata.get('label','?')}]: {d.page_content[:200]}"
        for d in knowledge_docs
    )
    prompt = (
        'You are an MBTI classifier. Respond with ONLY JSON:\n'
        '{"type":"XXXX","ie":"I or E","ie_conf":0.0-1.0,'
        '"sn":"S or N","sn_conf":0.0-1.0,'
        '"tf":"T or F","tf_conf":0.0-1.0,'
        '"jp":"J or P","jp_conf":0.0-1.0}\n\n'
        f'Similar posts:\n{sim_str}\n\n'
        f'Knowledge:\n{know_str}\n\n'
        f'Posts:\n{posts_text[:1000]}\n\n'
        'Classify. Output ONLY JSON:'
    )
    raw = llm.invoke(prompt)
    parsed = parse_agent_result(str(raw))
    if parsed:
        parsed = validate_result(parsed)
        if CONFIG['GUARDRAIL_ENABLED']:
            predicted_dims = {k: parsed[k] for k in ['ie', 'sn', 'tf', 'jp']}
            guard_scores = guardrail_score_prediction(posts_text, predicted_dims)
            for k in ['ie', 'sn', 'tf', 'jp']:
                parsed[f'{k}_guard'] = guard_scores.get(k, 0.0)
        return parsed
    return {
        'type': 'INFP', 'ie': 'I', 'ie_conf': 0.5,
        'sn': 'N', 'sn_conf': 0.5, 'tf': 'F', 'tf_conf': 0.5,
        'jp': 'P', 'jp_conf': 0.5,
    }


# ═══════════════════════════════════════════════════════════════
#  DEMO: LangChain Agent Trace
# ═══════════════════════════════════════════════════════════════
DEMO_INDICES = [0, 50, 100]
NUM_DEMOS = len(DEMO_INDICES)

# Reset memory
_pattern_memory = {}
_experience_store = []
_experience_faiss_index = None
_total_memories_stored = 0

print('=' * 70)
print(f'🔬 DEMO: {NUM_DEMOS} Users — LangChain ReAct Agent Trace')
print('=' * 70)
print(f'   Agent: LangChain AgentExecutor + create_react_agent')
print(f'   Tools: {[t.name for t in tools]}')
if CONFIG['GUARDRAIL_ENABLED']:
    print(f'   Guardrail: NLI Post-Verification ({CONFIG["NLI_MODEL"]}) — '
          f'reject if P(entailment) < {CONFIG["GUARDRAIL_THRESHOLD"]} '
          f'(max {CONFIG["MAX_GUARDRAIL_RETRIES"]} retries)')
else:
    print(f'   Guardrail: OFF')
print(f'   Long-term memory starts EMPTY — watch it grow!\n')

for demo_num, sample_idx in enumerate(DEMO_INDICES, 1):
    if sample_idx >= len(test_df):
        print(f'\n⚠️  Sample index {sample_idx} out of range, skipping.')
        continue

    row = test_df.iloc[sample_idx]
    posts_text = row['concat_posts']
    query_emb  = test_embeddings_np[sample_idx]

    true_type = row[LABEL_COL].upper()
    true_dims = {
        'ie': DIM_LETTER_MAP_INV['ie'][row['label_ie']],
        'sn': DIM_LETTER_MAP_INV['sn'][row['label_sn']],
        'tf': DIM_LETTER_MAP_INV['tf'][row['label_tf']],
        'jp': DIM_LETTER_MAP_INV['jp'][row['label_jp']],
    }

    t0 = time.time()
    parsed, output, steps = run_agent_on_sample(posts_text, query_emb, verbose=True)
    elapsed = time.time() - t0

    if parsed is None:
        parsed = fallback_predict(posts_text, query_emb)

    pred_type = parsed.get('type', '????')

    print(f'\n{"=" * 70}')
    print(f'  🧑 DEMO {demo_num}/{NUM_DEMOS}  |  Test Sample #{sample_idx}')
    print(f'{"=" * 70}')

    preview = posts_text[:300].replace('\n', ' ')
    print(f'\n📝 Posts preview:\n   "{preview}..."')

    match_icon = '✅' if pred_type == true_type else '❌'
    print(f'\n📋 True MBTI:      {true_type}')
    print(f'🤖 Predicted MBTI: {pred_type}  {match_icon}')
    print(f'⏱  Inference time: {elapsed:.1f}s')

    # ── Show guardrail activity ──
    if CONFIG['GUARDRAIL_ENABLED']:
        guardrail_rejections = sum(
            1 for a, obs in steps
            if hasattr(a, 'tool') and a.tool == 'submit_prediction'
            and 'PREDICTION REJECTED' in str(obs)
        )
        if guardrail_rejections > 0:
            print(f'🛡️  NLI rejections: {guardrail_rejections} '
                  f'(P(entailment) below threshold — agent revised)')

    print(f'\n📊 Axis Predictions:')
    dim_full = {'ie': 'I/E', 'sn': 'S/N', 'tf': 'T/F', 'jp': 'J/P'}
    for key in ['ie', 'sn', 'tf', 'jp']:
        pred_letter = parsed.get(key, '?')
        conf = parsed.get(f'{key}_conf', 0.0)
        guard_score = parsed.get(f'{key}_guard', -1)
        true_letter = true_dims[key]
        axis_match = '✅' if pred_letter == true_letter else '❌'
        conf_bar = '█' * int(conf * 20) + '░' * (20 - int(conf * 20))
        guard_str = f' P(ent)={guard_score:.2f}' if guard_score >= 0 else ''
        print(f'   {dim_full[key]}: {pred_letter} (conf={conf:.2f}{guard_str}) '
              f'[{conf_bar}] true={true_letter} {axis_match}')

    # ── NLI Post-Verification Analysis (independent 3rd-party) ──
    if CONFIG['GUARDRAIL_ENABLED']:
        predicted_dims = {k: parsed.get(k, '?') for k in ['ie', 'sn', 'tf', 'jp']}
        _, eval_details = guardrail_score_prediction(posts_text, predicted_dims, detailed=True)
        threshold = CONFIG['GUARDRAIL_THRESHOLD']

        print(f'\n🛡️ NLI Post-Verification Analysis (Independent 3rd-Party):')
        print(f'   Model: {CONFIG["NLI_MODEL"]}')
        print(f'   Threshold: P(entailment) ≥ {threshold}')
        print(f'   ┌───────┬──────┬──────────┬──────────┬──────────┬───────────────┐')
        print(f'   │  Dim  │ Pred │ P(ent)   │ P(neu)   │ P(con)   │    Verdict    │')
        print(f'   ├───────┼──────┼──────────┼──────────┼──────────┼───────────────┤')
        for key in ['ie', 'sn', 'tf', 'jp']:
            d = eval_details.get(key, {})
            pred_l = d.get('predicted', '?')
            p_ent = d.get('p_entail', 0)
            p_neu = d.get('p_neutral', 0)
            p_con = d.get('p_contra', 0)
            label = d.get('label', '?')
            vi = '✅' if label == 'ENTAILMENT' else '❌' if label == 'CONTRADICTION' else '⚠️'
            print(f'   │ {dim_full[key]:>5} │  {pred_l}   │  {p_ent:.4f}  │  {p_neu:.4f}  │  {p_con:.4f}  │ {vi} {label:<11} │')
        print(f'   └───────┴──────┴──────────┴──────────┴──────────┴───────────────┘')

        weak_dims = [dim_full[k] for k in ['ie', 'sn', 'tf', 'jp']
                     if eval_details.get(k, {}).get('score', 0) < threshold]
        if weak_dims:
            print(f'   ⛔ Weak dims: {"_".join(weak_dims)} '
                  f'(P(entailment) < {threshold})')
        else:
            print(f'   ✅ All dimensions pass entailment threshold')

        print(f'\n🔎 NLI Verification Details:')
        for key in ['ie', 'sn', 'tf', 'jp']:
            d = eval_details.get(key, {})
            pred_l = d.get('predicted', '?')
            p_ent = d.get('p_entail', 0)
            p_con = d.get('p_contra', 0)
            hyp = EVAL_HYPOTHESES.get(key, {}).get(pred_l, '?')
            print(f'\n   {dim_full[key]} [predicted {pred_l}]:')
            print(f'     hypothesis: "{hyp}"')
            print(f'     → P(entailment)={p_ent:.4f}  P(contradiction)={p_con:.4f}  '
                  f'(threshold={threshold})')

    # ── Agent trace ──
    print(f'\n🔄 LangChain Agent Trace ({len(steps)} tool calls):')
    for i, (action, observation) in enumerate(steps):
        tool_name = action.tool if hasattr(action, 'tool') else str(action)
        tool_input = action.tool_input if hasattr(action, 'tool_input') else ''
        log = action.log if hasattr(action, 'log') else ''
        icons = {
            'analyze_text': '🔬', 'retrieve_similar_posts': '🔍',
            'retrieve_mbti_knowledge': '📚', 'recall_experience': '💡',
            'submit_prediction': '🧠',
        }
        icon = icons.get(tool_name, '▶')

        is_rejected = (tool_name == 'submit_prediction'
                       and 'PREDICTION REJECTED' in str(observation))
        reject_tag = ' 🛡️ REJECTED' if is_rejected else ''

        print(f'\n   Step {i+1}: {icon} [{tool_name}]{reject_tag}')
        thought_match = re.search(r'Thought:\s*(.+?)(?=\nAction)', log, re.DOTALL)
        if thought_match:
            print(f'   💭 Thought: {thought_match.group(1).strip()[:400]}')
        if tool_input:
            print(f'   🔧 Input: {str(tool_input)[:200]}')
        print(f'   👁  Observation: {str(observation)[:300]}')

    trace_text = ' → '.join(
        f"{a.tool}({str(a.tool_input)[:50]})"
        for a, _ in steps if hasattr(a, 'tool')
    )
    stored = store_experience(query_emb, posts_text, parsed, trace_text)
    _pattern_memory[pred_type] = _pattern_memory.get(pred_type, 0) + 1
    mem_icon = '✅' if stored else '⏭'
    print(f'\n   💾 Memory: pattern["{pred_type}"]={_pattern_memory[pred_type]}, '
          f'{mem_icon} experience store size={len(_experience_store)}')
    print(f'{"─" * 70}')

print(f'\n{"=" * 70}')
print(f'📊 Long-term Memory Summary after {NUM_DEMOS} Demos:')
print(f'   Experiences stored:  {len(_experience_store)}/{NUM_DEMOS}')
print(f'   Type distribution:   '
      f'{dict(sorted(_pattern_memory.items(), key=lambda x: -x[1]))}')
print(f'{"=" * 70}')

# Reset for full run

_pattern_memory = {}print(f'\n🔬 Demo complete. Memory reset for full test run.')

_experience_store = []_total_memories_stored = 0
_experience_faiss_index = None

## 🚀 Cell 9 — Run Agent Inference on Test Set

Process all test samples through the LangChain `AgentExecutor`. Each sample runs the full ReAct loop autonomously.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  RUN LANGCHAIN AGENT INFERENCE ON TEST SET
# ═══════════════════════════════════════════════════════════════
n_test = len(test_df)

print(f'⏳ [RAG+Agent] LangChain AgentExecutor on {n_test} test samples...')
print(f'   Agent:           create_react_agent (ReAct)')
print(f'   Max iterations:  {CONFIG["MAX_AGENT_ITERATIONS"]}')
print(f'   Tools:           {[t.name for t in tools]}')
if CONFIG['GUARDRAIL_ENABLED']:
    print(f'   Guardrail:       NLI Post-Verification ({CONFIG["NLI_MODEL"]}) — '
          f'reject if P(entailment) < {CONFIG["GUARDRAIL_THRESHOLD"]}')
else:
    print(f'   Guardrail:       OFF')
print(f'   VectorStores:    post_vectorstore + knowledge_vectorstore\n')

agent_results = []
errors = 0
agent_fallbacks = 0
guardrail_total_rejections = 0
t_start = time.time()

for i in tqdm(range(n_test), desc='Agent inference'):
    posts_text = test_df.iloc[i]['concat_posts']
    query_emb = test_embeddings_np[i]

    try:
        parsed, output, steps = run_agent_on_sample(posts_text, query_emb, verbose=False)

        if CONFIG['GUARDRAIL_ENABLED']:
            for a, obs in steps:
                if hasattr(a, 'tool') and a.tool == 'submit_prediction':
                    if 'PREDICTION REJECTED' in str(obs):
                        guardrail_total_rejections += 1

        if parsed is None:
            parsed = fallback_predict(posts_text, query_emb)
            agent_fallbacks += 1

        agent_results.append(parsed)

        mbti = parsed.get('type', 'INFP')
        _pattern_memory[mbti] = _pattern_memory.get(mbti, 0) + 1
        trace_text = ' → '.join(
            f"{a.tool}" for a, _ in steps if hasattr(a, 'tool')
        )
        store_experience(query_emb, posts_text, parsed, trace_text)

    except Exception as e:
        errors += 1
        agent_results.append({
            'type': 'INFP', 'ie': 'I', 'ie_conf': 0.5,
            'sn': 'N', 'sn_conf': 0.5, 'tf': 'F', 'tf_conf': 0.5,
            'jp': 'P', 'jp_conf': 0.5,
        })
        if errors <= 5:
            print(f'  ⚠️ Sample {i} error: {str(e)[:200]}')

    if (i + 1) % 50 == 0:
        elapsed = time.time() - t_start
        done = len(agent_results)
        speed = done / elapsed
        eta = (n_test - done) / speed if speed > 0 else 0
        top_types = sorted(_pattern_memory.items(), key=lambda x: -x[1])[:5]
        print(f'  [{done}/{n_test}] '
              f'speed={speed:.2f} samples/s | ETA={eta/60:.1f}min | '
              f'errors={errors} | fallbacks={agent_fallbacks}'
              + (f' | nli_rejections={guardrail_total_rejections}' if CONFIG['GUARDRAIL_ENABLED'] else ''))

elapsed_total = time.time() - t_start
print(f'\n✅ Inference complete in {elapsed_total/60:.1f} minutes')
print(f'   Speed: {n_test/elapsed_total:.2f} samples/s')
print(f'   Errors: {errors}/{n_test}')
print(f'   Fallbacks: {agent_fallbacks}')
if CONFIG['GUARDRAIL_ENABLED']:
    print(f'   NLI rejections: {guardrail_total_rejections} '
          f'(P(entailment) below threshold — agent revised)')
else:
    print(f'   Guardrail: OFF (no verification)')
print(f'   Experience store: {len(_experience_store)} '
      f'({len(_experience_store)/max(n_test,1)*100:.1f}%)')
print(f'   Type distribution:')
for t, cnt in sorted(_pattern_memory.items(), key=lambda x: -x[1]):
    print(f'     {t}: {cnt} ({cnt/n_test*100:.1f}%)')

## 📊 Cell 10 — Metrics (F1, Accuracy, AUC) + Save CSV

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  METRICS: F1, Accuracy, AUC  —  4 axes + full 16 types
# ═══════════════════════════════════════════════════════════════

DIM_LABEL_MAP = {
    'ie': {'I': 1, 'E': 0},
    'sn': {'N': 1, 'S': 0},
    'tf': {'F': 1, 'T': 0},
    'jp': {'J': 1, 'P': 0},
}
DIM_KEYS = ['ie', 'sn', 'tf', 'jp']

y_true_axes = test_df[LABEL_COLS].values
y_pred_axes = np.zeros((len(agent_results), 4), dtype=int)
y_prob_axes = np.zeros((len(agent_results), 4), dtype=float)

for i, res in enumerate(agent_results):
    for j, key in enumerate(DIM_KEYS):
        letter = res.get(key, 'I' if j == 0 else 'N' if j == 1 else 'F' if j == 2 else 'P')
        pred   = DIM_LABEL_MAP[key].get(letter, 0)
        conf   = res.get(f'{key}_conf', 0.5)
        y_pred_axes[i, j] = pred
        y_prob_axes[i, j] = conf if pred == 1 else (1.0 - conf)


# ═══════════════ 4-AXIS METRICS ═══════════════
print('=' * 70)
print('📊  4-AXIS METRICS — RAG + LLM Agent (LangChain)')
print('=' * 70)
print(f'{"Axis":<8} {"F1-Macro":<12} {"Accuracy":<12} {"AUC-ROC":<12}')
print('-' * 44)

f1_list, acc_list, auc_list = [], [], []
for i, name in enumerate(DIM_NAMES):
    f1  = f1_score(y_true_axes[:, i], y_pred_axes[:, i], average='macro', zero_division=0)
    acc = accuracy_score(y_true_axes[:, i], y_pred_axes[:, i])
    try:
        auc = roc_auc_score(y_true_axes[:, i], y_prob_axes[:, i])
    except ValueError:
        auc = 0.5
    f1_list.append(f1); acc_list.append(acc); auc_list.append(auc)
    print(f'{name:<8} {f1:<12.4f} {acc:<12.4f} {auc:<12.4f}')

print('-' * 44)
print(f'{"Average":<8} {np.mean(f1_list):<12.4f} {np.mean(acc_list):<12.4f} '
      f'{np.mean(auc_list):<12.4f}')


# ═══════════════ 16-TYPE METRICS ═══════════════
y_true_type = test_df[LABEL_COL].str.upper().tolist()
y_pred_type = [r['type'].upper() for r in agent_results]

type_to_idx = {t.upper(): i for i, t in enumerate(MBTI_TYPES)}
y_true_16 = np.array([type_to_idx.get(t, 0) for t in y_true_type])
y_pred_16 = np.array([type_to_idx.get(t, 0) for t in y_pred_type])

y_prob_16 = np.zeros((len(agent_results), 16), dtype=float)
for i in range(len(agent_results)):
    for t_idx, t_name in enumerate(MBTI_TYPES):
        t_upper = t_name.upper()
        p = 1.0
        for j, (key, letter) in enumerate(zip(DIM_KEYS, t_upper)):
            target = DIM_LABEL_MAP[key].get(letter, 0)
            p *= y_prob_axes[i, j] if target == 1 else (1.0 - y_prob_axes[i, j])
        y_prob_16[i, t_idx] = p
    row_sum = y_prob_16[i].sum()
    if row_sum > 0:
        y_prob_16[i] /= row_sum

f1_16_macro    = f1_score(y_true_16, y_pred_16, average='macro', zero_division=0)
f1_16_weighted = f1_score(y_true_16, y_pred_16, average='weighted', zero_division=0)
acc_16         = accuracy_score(y_true_16, y_pred_16)

try:
    y_true_16_bin = label_binarize(y_true_16, classes=list(range(16)))
    auc_16 = roc_auc_score(y_true_16_bin, y_prob_16, average='macro', multi_class='ovr')
except Exception:
    auc_16 = 0.0

print(f'\n{"=" * 70}')
print('📊  16-TYPE METRICS — RAG + LLM Agent (LangChain)')
print(f'{"=" * 70}')
print(f'  Accuracy:            {acc_16:.4f}')
print(f'  F1 (Macro):          {f1_16_macro:.4f}')
print(f'  F1 (Weighted):       {f1_16_weighted:.4f}')
print(f'  AUC-ROC (Macro OvR): {auc_16:.4f}')

present_labels = sorted(set(y_true_16.tolist()) | set(y_pred_16.tolist()))
target_names   = [MBTI_TYPES[i].upper() for i in present_labels]
print(f'\n📋 Per-Type Classification Report:')
print(classification_report(
    y_true_16, y_pred_16,
    labels=present_labels,
    target_names=target_names,
    zero_division=0
))


# ═══════════════ SAVE TO CSV ═══════════════
agent_save = pd.DataFrame()
for i, c in enumerate(LABEL_COLS):
    agent_save[f'y_true_{c}'] = y_true_axes[:, i]
    agent_save[f'y_pred_{c}'] = y_pred_axes[:, i]
    agent_save[f'y_prob_{c}'] = np.round(y_prob_axes[:, i], 4)

agent_save['y_true_type'] = y_true_type
agent_save['y_pred_type'] = y_pred_type

agent_save.to_csv(f"{CONFIG['RESULT_DIR']}/rag_agent_predictions.csv", index=False)
print(f'\n✅ Saved → {CONFIG["RESULT_DIR"]}/rag_agent_predictions.csv')